### Preprocesamiento y Preparación de Datos para el Modelado

**Objetivo**
Transformar el dataset (`dataset_samur_modelado.csv`, de 1.320 filas y 75 columnas) en matrices numéricas limpias, sin valores ausentes, libres de multicolinealidad y correctamente particionadas para entrenar y evaluar modelos de Machine Learning (tanto lineales/paramétricos como basados en árboles).

**Política de inmutabilidad de datos:**
Todas las transformaciones se realizan sobre una **copia independiente** del dataset consolidado. El fichero original (`dataset_samur_modelado.csv`, con sus 1.320 filas y 146.567 intervenciones) permanece intacto para preservar la trazabilidad completa del proyecto.

**4 decisiones técnicas del preprocesamiento:**
1. **Acotación operativa del dominio (Filtrado de registros no informativos):** Delimitamos el entrenamiento a los 21 distritos administrativos oficiales de Madrid y a las 4 franjas horarias reales del día ($12 \times 21 \times 4 = 1.008$ observaciones). De este modo evitamos imputar datos socioeconómicos ficticios a ubicaciones desconocidas (`OTROS`) y eliminamos el ruido de los avisos sin hora registrada (`Desconocido`), reteniendo el **98,62 %** de las emergencias reales de 2024 sin valores nulos.
2. **Control de multicolinealidad (Eliminación de identificadores y variables redundantes):** Suprimimos códigos nominales sin valor predictivo (como el código postal o el ID del distrito) y descartamos variables brutas que ya fueron transformadas en métricas normalizadas o cíclicas durante la ingeniería de variables (por ejemplo, conservamos `mes_sin`/`mes_cos` en lugar de `mes_num`, y las tasas porcentuales de calendario en lugar de los conteos brutos de días). Esto estabiliza los coeficientes de los modelos lineales y evita redundancia de información.
3. **Codificación de variables categóricas (One-Hot Encoding):** Transformamos las variables cualitativas (`distrito` y `franja_horaria`) en indicadores binarios (`0` y `1`) descartando una categoría de referencia por variable (`drop_first=True`) para prevenir la colinealidad perfecta.
4. **Partición estratificada y escalado sin fuga de datos (*Data Leakage*):** Dividimos los datos en un conjunto de entrenamiento (train) y otro de evaluación (test) manteniendo exactamente la misma proporción de distritos y franjas en ambos subconjuntos. Posteriormente, estandarizamos las variables numéricas ajustando la escala **únicamente** con los datos de entrenamiento antes de aplicarla al conjunto de test.

#### 1. Decisión 1: Acotación Operativa del Dominio (Filtrado de Registros No Informativos)

Sobre una copia de `dataset_samur_modelado.csv`, se filtran los registros para conservar exclusivamente los **21 distritos administrativos oficiales de Madrid** y las **4 franjas horarias reales** (`Madrugada`, `Mañana`, `Tarde` y `Noche`). Esto reduce el conjunto de datos de **1.320 a 1.008 filas** ($12\text{ meses} \times 21\text{ distritos} \times 4\text{ franjas}$).

**¿Por qué?**
Esta decisión no se toma simplemente para eliminar los datos que faltan, sino para definir de forma clara qué problema queremos estudiar y qué datos necesitamos para ello:

1. **Coherencia con el objetivo del estudio:** El reto consiste en anticipar la demanda asistencial del SAMUR en cada distrito de Madrid para apoyar la planificación operativa de recursos. La categoría `'OTROS / DISTRITO NO IDENTIFICADO'` (60 filas, 601 intervenciones, apenas el **0,41 %** del total) agrupa incidentes ocurridos fuera del término municipal o sin ubicación registrada. Al no corresponder a ningún distrito real, carece de población, superficie, hospitales o indicadores socioeconómicos.
2. **Prevención de datos sintéticos ficticios (rechazo a la imputación):** Si mantuviéramos esas 60 filas y rellenáramos sus 17 variables territoriales con la media o mediana de Madrid, estaríamos inventando atributos demográficos inexistentes para un territorio indefinido, introduciendo sesgo y ruido en el aprendizaje de los coeficientes territoriales.
3. **Eliminación de ruido de registro en la dimensión horaria:** La franja `'Desconocido'` (252 filas, **1.422 intervenciones**, el **0,97 %** del total) no representa un quinto tramo horario del día, sino llamadas en las que no quedó consignada la hora de activación. Predecir "cuántos fallos de registro horario habrá" carece de utilidad para la asignación de ambulancias por turnos y distorsionaría la curva intradiaria real.
4. **Cobertura casi total y matriz 100 % densa:** Al acotar el dataset a las **1.008 combinaciones operativas**, se conserva el **98,63 % de todas las emergencias reales de 2024** (`144.544` de `146.567` intervenciones) y se garantiza una matriz con **0 valores nulos** en todas sus columnas, permitiendo comparar en igualdad de condiciones modelos lineales, de conteo y de árboles.

In [1]:
import os
import pandas as pd
import numpy as np

# Cargar el dataset consolidado sin modificarlo
ruta_origen = os.path.join('datasets limpios', 'dataset_samur_modelado.csv')
df_original = pd.read_csv(ruta_origen)

# Trabajar sobre una copia independiente
df_prep = df_original.copy()

# 1. Filtrado de registros no operativos
df_prep = df_prep[
    (df_prep['distrito'] != 'OTROS / DISTRITO NO IDENTIFICADO') &
    (df_prep['franja_horaria'] != 'Desconocido')
].copy()

# Auditoría de la Decisión 1
intervenciones_totales = df_original['total_intervenciones'].sum()
intervenciones_conservadas = df_prep['total_intervenciones'].sum()

print("AUDITORÍA DECISIÓN 1: FILTRADO OPERATIVO")
print("=" * 55)
print(f"Filas originales:           {len(df_original)}")
print(f"Filas tras filtrado:        {len(df_prep)} (12 meses x 21 distritos x 4 franjas)")
print(f"Intervenciones conservadas: {intervenciones_conservadas:,} de {intervenciones_totales:,} "
      f"({intervenciones_conservadas / intervenciones_totales * 100:.2f}%)")
print(f"Valores nulos restantes:    {df_prep.isna().sum().sum()}")

assert len(df_prep) == 12 * 21 * 4, "Error: se esperaban exactamente 1.008 filas."
assert df_prep.isna().sum().sum() == 0, "Error: siguen existiendo valores nulos."

AUDITORÍA DECISIÓN 1: FILTRADO OPERATIVO
Filas originales:           1320
Filas tras filtrado:        1008 (12 meses x 21 distritos x 4 franjas)
Intervenciones conservadas: 144,559 de 146,567 (98.63%)
Valores nulos restantes:    0


#### 2. Decisión 2: Control de Multicolinealidad (Eliminación de Identificadores y Redundancias)

Se eliminan **9 columnas** del conjunto de predictores que pertenecen a dos grupos:
* **Identificadores nominales o arbitrarios (3 variables):** `mes` (texto), `distrito_id` (código numérico del 1 al 21) y `codigo_postal_principal`.
* **Variables temporales y de calendario redundantes (6 variables):** `mes_num`, `dias_mes`, `dias_festivos`, `dias_fin_semana`, `dias_con_evento` y `dias_vacaciones`.

**¿Por qué?**
1. **Evitar relaciones numéricas espurias:** Variables como `distrito_id` (donde Centro es `1` y Barajas es `21`) o `codigo_postal_principal` (`28001`, `28002`...) son etiquetas administrativas representadas como números. Si se dejan como variables numéricas, los modelos de regresión interpretarían erróneamente que Barajas es "21 veces mayor" que Centro. La identidad de cada distrito ya queda recogida de forma rigurosa mediante One-Hot Encoding en el siguiente paso.
2. **Eliminar la multicolinealidad perfecta e inflada:** Durante la fase de ingeniería de variables transformamos `mes_num` en sus componentes trigonométricas cíclicas (`mes_sin` y `mes_cos`) para preservar la continuidad entre diciembre y enero, y normalizamos los conteos de días de calendario dividiéndolos por `dias_mes` para obtener porcentajes comparables (`pct_festivos`, `pct_fines_semana`, `pct_dias_evento` y `pct_vacaciones`). Mantener simultáneamente la variable original y su transformación provoca **colinealidad lineal**, lo que inestabiliza los coeficientes ($\beta$) y los errores estándar en regresiones lineales y de Poisson sin aportar información nueva.

In [2]:
# 2. Eliminación de identificadores y variables redundantes
columnas_eliminar = [
    'mes',                      # Redundante con mes_sin / mes_cos
    'mes_num',                  # Redundante con mes_sin / mes_cos
    'distrito_id',              # Código administrativo arbitrario
    'codigo_postal_principal',  # Código nominal sin orden cuantitativo
    'dias_mes',                 # Denominador ya integrado en las tasas porcentuales
    'dias_festivos',            # Redundante con pct_festivos
    'dias_fin_semana',          # Redundante con pct_fines_semana
    'dias_con_evento',          # Redundante con pct_dias_evento
    'dias_vacaciones'           # Redundante con pct_vacaciones
]

df_prep = df_prep.drop(columns=columnas_eliminar)

print("AUDITORÍA DECISIÓN 2: ELIMINACIÓN DE REDUNDANCIAS")
print("=" * 55)
print(f"Columnas eliminadas: {len(columnas_eliminar)}")
print(f"Columnas restantes:  {df_prep.shape[1]} (65 predictores + 1 variable objetivo)")

AUDITORÍA DECISIÓN 2: ELIMINACIÓN DE REDUNDANCIAS
Columnas eliminadas: 9
Columnas restantes:  66 (65 predictores + 1 variable objetivo)


#### 3. Decisión 3: Codificación de Variables Categóricas (One-Hot Encoding)

Se separa la matriz de variables explicativas ($X$) de la variable objetivo ($y = \text{total\_intervenciones}$) y se transforman las dos variables categóricas restantes (`distrito`, con 21 niveles, y `franja_horaria`, con 4 niveles) en variables binarias (`0` y `1`) mediante `pd.get_dummies(..., drop_first=True, dtype=int)`.

**¿Por qué?**
Los algoritmos matemáticos de Machine Learning requieren entradas exclusivamente numéricas. Al aplicar One-Hot Encoding creamos una columna indicadora por cada distrito y franja horaria sin imponer ningún orden artificial entre categorías.
* **Prevención de la trampa de las variables ficticias (Dummy Variable Trap):** Al utilizar `drop_first=True`, eliminamos la primera categoría de cada variable (Arganzuela en distrito y Madrugada en franja horaria). Esto se hace porque no necesitamos una columna para cada categoría: si todas las demás tienen valor 0, ya sabemos que pertenece a la categoría que hemos eliminado. Por tanto, esa categoría pasa a ser la **categoría de referencia**, respecto a la cual se comparan las demás. Así evitamos tener columnas que contienen información repetida y que pueden generar un problema de multicolinealidad en los modelos de regresión.

In [3]:
# Guardamos la combinación distrito + franja para estratificar en el paso siguiente
estrato = df_prep['distrito'] + "_" + df_prep['franja_horaria']

# 3. Separación de predictores (X) y variable objetivo (y)
X = df_prep.drop(columns=['total_intervenciones'])
y = df_prep['total_intervenciones']

# Codificación One-Hot evitando colinealidad perfecta (drop_first=True)
X_encoded = pd.get_dummies(
    X,
    columns=['distrito', 'franja_horaria'],
    drop_first=True,
    dtype=int
)

print("AUDITORÍA DECISIÓN 3: ONE-HOT ENCODING")
print("=" * 55)
print(f"Columnas antes del encoding: {X.shape[1]}")
print(f"Columnas tras el encoding:   {X_encoded.shape[1]} "
      f"(63 numéricas + 20 dummies de distrito + 3 dummies de franja)")
print(f"Tipos de datos no numéricos restantes: {X_encoded.select_dtypes(exclude='number').shape[1]}")

assert X_encoded.select_dtypes(exclude='number').shape[1] == 0, "Error: quedan columnas no numéricas."

AUDITORÍA DECISIÓN 3: ONE-HOT ENCODING
Columnas antes del encoding: 65
Columnas tras el encoding:   86 (63 numéricas + 20 dummies de distrito + 3 dummies de franja)
Tipos de datos no numéricos restantes: 0


#### 4. Decisión 4: Estrategia de Partición (Train/Test Split) y Escalado sin Fuga de Datos

**¿Qué se hace?**
Se divide el conjunto de datos en un **80 % para entrenamiento** (`806` observaciones) y un **20 % para evaluación final** (`202` observaciones) mediante **muestreo aleatorio estratificado** sobre la clave combinada `distrito × franja_horaria`. Posteriormente, se aplica una estandarización Z-score (`StandardScaler`) ajustando la media ($\mu$) y la desviación típica ($\sigma$) exclusivamente sobre `X_train` para transformar después ambos conjuntos (`X_train_scaled` y `X_test_scaled`).

**Justificación de nuestra decisión**

1. **Elección del método de reparto (Estratificado vs. Aleatorio puro vs. Temporal):**
   * *Por qué se descarta el reparto aleatorio simple:* El dataset contiene 84 subgrupos operativos únicos (`21 distritos × 4 franjas horarias`), con volúmenes muy asimétricos entre sí (por ejemplo, `CENTRO` en `Tarde` frente a `BARAJAS` en `Madrugada`). Un reparto puramente aleatorio podría concentrar por azar los meses pico de distritos grandes en test o en train, distorsionando la evaluación.
   * *Por qué se descarta un corte cronológico (ej. últimos 2 meses para test):* Al contar únicamente con un ciclo anual completo (año 2024), reservar los últimos meses del año para evaluación privaría al modelo de aprender el comportamiento invernal navideño, rompiendo la representación del ciclo estacional completo.
   * *Ventaja del muestreo estratificado por `distrito + franja_horaria`:* Garantiza que las 84 combinaciones espacio-temporales tengan exactamente la misma representación proporcional en entrenamiento (~9–10 meses por pareja) y en evaluación (~2–3 meses por pareja), permitiendo al modelo aprender la estacionalidad completa de Madrid sin sesgos territoriales ni horarios.

2. **Elección de la proporción 80 % / 20 %:**
   * Para un conjunto de datos tabular de tamaño medio (N = 1.008 observaciones y 86 predictores tras codificar), la partición 80/20 ofrece el equilibrio óptimo entre capacidad de aprendizaje y fiabilidad de evaluación: proporciona 806 registros suficientes para estimar modelos con múltiples variables sin sobreajuste (overfitting) y reserva 202 registros independientes para medir la capacidad de generalización con baja varianza. Además, en la fase de modelado se complementará con validación cruzada (5-Fold Cross-Validation) dentro del conjunto de entrenamiento.

3. **Prevención de fuga de datos (Data Leakage) y doble salida:**
   * Las variables presentan magnitudes muy heterogéneas (desde proporciones hasta cientos de miles de habitantes). Ajustar el `StandardScaler` **después** de dividir y únicamente con los estadísticos de `X_train` impide que información de la distribución de `X_test` contamine el entrenamiento. Se conservan tanto las matrices sin escalar (`X_train`, `X_test`) para modelos basados en árboles como las estandarizadas (`X_train_scaled`, `X_test_scaled`) para modelos lineales y de conteo.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 4. División Train / Test estratificada (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=estrato
)

# Escalado estándar ajustado EXCLUSIVAMENTE sobre X_train (sin data leakage)
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("AUDITORÍA DECISIÓN 4: SPLIT ESTRATIFICADO Y ESCALADO")
print("=" * 55)
print(f"X_train / y_train: {X_train.shape} / {y_train.shape}")
print(f"X_test  / y_test:  {X_test.shape} / {y_test.shape}")
print(f"Media de población en X_train_scaled: {X_train_scaled['poblacion'].mean():.4f} (esperado: ~0)")
print(f"Desv. típica población en X_train_scaled: {X_train_scaled['poblacion'].std(ddof=0):.4f} (esperado: ~1)")

AUDITORÍA DECISIÓN 4: SPLIT ESTRATIFICADO Y ESCALADO
X_train / y_train: (806, 86) / (806,)
X_test  / y_test:  (202, 86) / (202,)
Media de población en X_train_scaled: -0.0000 (esperado: ~0)
Desv. típica población en X_train_scaled: 1.0000 (esperado: ~1)


#### 5. Exportación de los Conjuntos Listos para el Modelado

Creamos ahora un subdirectorio específico (`datasets modelado/`) y se guardan en formato CSV tanto las matrices sin escalar (`X_train`, `X_test`), como las estandarizadas (`X_train_scaled`, `X_test_scaled`) y sus vectores objetivo (`y_train`, `y_test`), manteniendo intacto el fichero original.

Esto nos permite iniciar la siguiente fase en un notebook limpio cargando directamente las particiones exactas ya validadas, asegurando que todos los algoritmos se entrenen y evalúen bajo exactamente las mismas observaciones y condiciones.

In [6]:
import os

# Definir la ruta base del proyecto y crear la nueva carpeta dedicada al modelo
ruta_proyecto = r"C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias"
carpeta_salida = os.path.join(ruta_proyecto, 'datasets_modelos')

os.makedirs(carpeta_salida, exist_ok=True)

# Guardar conjuntos sin escalar (para árboles: Random Forest, XGBoost, LightGBM)
X_train.to_csv(os.path.join(carpeta_salida, 'X_train.csv'), index=False, encoding='utf-8-sig')
X_test.to_csv(os.path.join(carpeta_salida, 'X_test.csv'), index=False, encoding='utf-8-sig')

# Guardar conjuntos escalados (para modelos lineales y de conteo)
X_train_scaled.to_csv(os.path.join(carpeta_salida, 'X_train_scaled.csv'), index=False, encoding='utf-8-sig')
X_test_scaled.to_csv(os.path.join(carpeta_salida, 'X_test_scaled.csv'), index=False, encoding='utf-8-sig')

# Guardar variable objetivo
y_train.to_csv(os.path.join(carpeta_salida, 'y_train.csv'), index=False, encoding='utf-8-sig')
y_test.to_csv(os.path.join(carpeta_salida, 'y_test.csv'), index=False, encoding='utf-8-sig')

print("EXPORTACIÓN COMPLETADA CON ÉXITO")
print("=" * 55)
print("Archivos guardados en:", carpeta_salida)
print("El dataset original en 'datasets limpios\\dataset_samur_modelado.csv' permanece intacto.")

EXPORTACIÓN COMPLETADA CON ÉXITO
Archivos guardados en: C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\datasets_modelos
El dataset original en 'datasets limpios\dataset_samur_modelado.csv' permanece intacto.
